# 02.1 Image Data and Transforms

Once you enter computer vision, two things must become clear first:

1. what an image looks like in `PyTorch`
2. what preprocessing steps are usually applied before feeding images into a model

Key concepts in this notebook:

- grayscale image
- RGB image
- channel
- image tensor shape
- batched images
- transforms
- normalization

## Learning Goals

After this notebook, you should be able to:

1. Understand the standard shape of image tensors.
2. Distinguish a single image from a batch of images.
3. Use `torchvision.transforms` to build a basic preprocessing pipeline.
4. Understand why scaling and normalization are common.
5. Make a custom dataset support transforms.
6. Prepare data input for later CNN classification tasks.

In [ ]:
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import load_digits
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


## What Shape Does One Image Have in `PyTorch`?

The most common image shape convention in `PyTorch` is:

- one grayscale image / one grayscale image: `(C, H, W)`, where `C=1`
- one RGB image / one RGB image: `(C, H, W)`, where `C=3`
- a batch of images: `(N, C, H, W)`

This differs from many other libraries that use `(H, W, C)`.


In [ ]:
digits = load_digits()
images = digits.images
labels = digits.target

print("images.shape =", images.shape)
print("labels.shape =", labels.shape)
print("one raw image shape / one raw image shape =", images[0].shape)

A single image from `sklearn digits` has shape `(8, 8)`, meaning it only has height and width.

Because it is grayscale, we usually explicitly add the channel dimension.


In [ ]:
raw_img = torch.tensor(images[0], dtype=torch.float32)
img_chw = raw_img.unsqueeze(0)

print("raw_img.shape =", raw_img.shape)
print("img_chw.shape =", img_chw.shape)

In [ ]:
plt.figure(figsize=(3, 3))
plt.imshow(raw_img, cmap="gray")
plt.title(f"Digit / digit: {labels[0]}")
plt.axis("off")
plt.show()

## Single Images vs Batches of Images

Models usually do not consume a single image; they consume a batch.

That means:

- one image: `(C, H, W)`
- batched images: `(N, C, H, W)`

In [ ]:
batch = torch.stack([
    torch.tensor(images[0], dtype=torch.float32).unsqueeze(0),
    torch.tensor(images[1], dtype=torch.float32).unsqueeze(0),
    torch.tensor(images[2], dtype=torch.float32).unsqueeze(0),
])

print("batch.shape =", batch.shape)

`batch.shape == (3, 1, 8, 8)` means:

- 3 images
- 1 channel per image
- height 8
- width 8

In [ ]:
# Exercise 1
# Turn the first 5 images in the digits dataset into one batch.
#Requirements / Requirements:
# 1. First convert each image to (1, 8, 8)
# 2. The final batch shape should be (5, 1, 8, 8)

# imgs =
# batch5 =
# print(batch5.shape)

In [ ]:
# Exercise 1 Reference Solution

imgs = [torch.tensor(images[i], dtype=torch.float32).unsqueeze(0) for i in range(5)]
batch5 = torch.stack(imgs)
print(batch5.shape)

## Why Use Transforms?

Images usually go through a sequence of preprocessing steps before entering a model.

Common reasons:

- scale pixel values into a reasonable range
- normalize the input
- apply data augmentation

This notebook first focuses on the first two.


In [ ]:
print("raw pixel range / raw pixel range:", raw_img.min().item(), raw_img.max().item())

The pixel values in the `digits` dataset roughly range from `0` to `16`.

normalization.  
A common practice is to first scale them into `0~1`, and then normalize them.

In [ ]:
preprocess = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

img_processed = preprocess(img_chw)

print("before preprocessing / before preprocessing:", img_chw.min().item(), img_chw.max().item())
print("after preprocessing / after preprocessing:", img_processed.min().item(), img_processed.max().item())
print("processed mean / processed mean:", img_processed.mean().item())

`Normalize(mean=[0.5], std=[0.5])` for a grayscale image means:

- Subtract `0.5`
- Then divide by `0.5`

If the input is already in `0~1`, the output will roughly fall in `-1~1`. 
If the input is already in `0~1`, the output will roughly land in `-1~1`.

## Making a Custom Dataset Support Transforms

This is very important because real projects usually look like this:

- a dataset class
- a `transform` argument
- apply the transform in `__getitem__`

In [ ]:
class DigitsImageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.labels[index], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        return image, label


dataset = DigitsImageDataset(images, labels, transform=preprocess)
img0, label0 = dataset[0]
print("img0.shape =", img0.shape)
print("label0 =", label0)
print("img0.dtype =", img0.dtype)

In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=False)
xb, yb = next(iter(loader))

print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)
print("xb.min() =", xb.min().item())
print("xb.max() =", xb.max().item())

In [ ]:
# Exercise 2
# Implement a transform that only scales values to 0~1, without Normalize.
#Then take dataset[0] and print the minimum and maximum values.
# Then inspect dataset[0] and print the min and max values.

# simple_transform =
# simple_dataset =
# img, label =
# print(img.min().item(), img.max().item())

In [ ]:
# Exercise 2 Reference Solution

simple_transform = transforms.Compose([
    transforms.Lambda(lambda x: x.float() / 16.0),
])
simple_dataset = DigitsImageDataset(images, labels, transform=simple_transform)
img, label = simple_dataset[0]
print(img.min().item(), img.max().item())

## About Data Augmentation

Common data augmentation for image tasks / common data augmentation includes:

- random crop
- random rotation
- color jitter
- horizontal flip

But be careful: more augmentation is not always better.

For digit recognition, horizontal flipping often changes the meaning of the digit.


## Summary

The most important goal of this notebook is to make image data format completely clear.

You should now be able to answer:

1. Why does `PyTorch` usually use `(C, H, W)` instead of `(H, W, C)`?
2. What is the difference between the shape of one image and a batch of images?
3. Why are images often scaled and then normalized?
4. Why is `transform` often built into the `Dataset`?

Suggested next step:

- Move to the CNN basics notebook and understand what convolution layers do to images.